# =============================================================================
# Alternate-Data Credit Underwriting for "Thin-File" MSMEs
# =============================================================================
# XGBoost/CatBoost with Monotonic Constraints + SHAP Explainability
# India-specific: UPI QR velocity, GST filing, telecom recharges, utility bills

In [1]:
!pip install faker

In [2]:
!pip install catboost

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("laotse/credit-risk-dataset")

In [5]:
# =============================================================================
# COMPLETE FIXED VERSION - Train on REAL DATA ONLY
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import xgboost as xgb
import os

np.random.seed(42)

DATASET_PATH = "/content/credit_risk_dataset.csv"  # Your filename

print(f"\n{'='*60}")
print("CREDIT UNDERWRITING - TRAIN ON REAL DATA")
print(f"{'='*60}")

# STEP 1: Load REAL data
print("\nLoading your dataset...")
df = pd.read_csv(DATASET_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Default rate: {df['loan_status'].mean():.1%}")

# STEP 2: Use DIRECT features (no transformation!)
# Use the original Kaggle features directly
feature_cols = [
    'person_age', 'person_income', 'person_emp_length',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income',
    'cb_person_default_on_file', 'cb_person_cred_hist_length'
]

# Convert loan_grade to numeric
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6}
df['loan_grade_numeric'] = df['loan_grade'].map(grade_map)
feature_cols.append('loan_grade_numeric')

# Convert home_ownership to numeric
home_map = {'RENT': 0, 'OWN': 1, 'MORTGAGE': 2}
df['home_ownership_num'] = df['person_home_ownership'].map(home_map)
feature_cols.append('home_ownership_num')

# Convert 'cb_person_default_on_file' from 'Y'/'N' to 1/0
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'Y': 1, 'N': 0})

print(f"\nUsing {len(feature_cols)} features: {feature_cols}")

X = df[feature_cols].values
y = df['loan_status'].values

# Fill missing values
X = np.nan_to_num(X, nan=0)

print(f"✓ Features: {X.shape}, Labels: {y.shape}")
print(f"  Default rate: {y.mean():.1%}")

# STEP 3: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=42
)

print(f"\nSplit: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

# STEP 4: Train XGBoost (NO monotonic constraints for real data)
print("\nTraining XGBoost on REAL data...")
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_cols)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=feature_cols)

model = xgb.train(
    {'objective': 'binary:logistic', 'eval_metric': 'auc', 'max_depth': 4, 'learning_rate': 0.1, 'seed': 42},
    dtrain, num_boost_round=100, evals=[(dtrain, 'train'), (dval, 'val')], verbose_eval=20
)

val_auc = roc_auc_score(y_val, model.predict(dval))
print(f"✓ Val AUC: {val_auc:.4f}")

model.save_model("credit_risk_model.pkl")

# STEP 5: Evaluate on test
test_pred = model.predict(xgb.DMatrix(X_test, feature_names=feature_cols))
test_auc = roc_auc_score(y_test, test_pred)
test_prec = precision_score(y_test, (test_pred > 0.5).astype(int))
test_rec = recall_score(y_test, (test_pred > 0.5).astype(int))

print(f"\n{'='*60}")
print("TEST SET RESULTS")
print(f"{'='*60}")
print(f"AUC: {test_auc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall: {test_rec:.4f}")

if test_auc > 0.75:
    print(f"✓ GOOD - Model works on real data!")
elif test_auc > 0.65:
    print(f"⚠ ACCEPTABLE - Model needs tuning")
else:
    print(f"❌ BAD - Model predicting randomly")

# STEP 6: Explain sample
print(f"\n{'='*60}")
print("SAMPLE EXPLANATION")
print(f"{'='*60}")

sample_idx = 0
dmat_sample = xgb.DMatrix(X_test[sample_idx:sample_idx+1], feature_names=feature_cols)
pd = model.predict(dmat_sample)[0]
shap_vals = model.predict(dmat_sample, pred_contribs=True)[0, :-1]

print(f"Customer {sample_idx}:")
print(f"  PD: {pd:.4f} ({pd*100:.2f}%)")
print(f"  Actual Default: {y_test[sample_idx]}")
print(f"  Prediction: {'DEFAULT' if pd > 0.5 else 'SAFE'}")

shap_dict = {f: v for f, v in zip(feature_cols, shap_vals)}
top = sorted(shap_dict.items(), key=lambda x: abs(x[1]), reverse=True)[:3]

print(f"\nTop 3 Risk Factors:")
for i, (feat, val) in enumerate(top, 1):
    print(f"  {i}. {feat}: {val:.4f} → {'increases risk' if val > 0 else 'decreases risk'}")

# STEP 7: Feature importance
print(f"\n{'='*60}")
print("FEATURE IMPORTANCE")
print(f"{'='*60}")

importance = model.get_score(importance_type='gain')
for feat, imp in sorted(importance.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {feat}: {imp:.4f}")

# STEP 8: Test cases with REAL model
print(f"\n{'='*60}")
print("TEST CASES (Real Model)")
print(f"{'='*60}")

def create_test_case(age, income, emp_len, loan_amnt, int_rate, percent_income, hist_default, cred_hist, grade='C', home='RENT'):
    grade_num = grade_map.get(grade, 3)
    home_num = home_map.get(home, 0)
    return np.array([[age, income, emp_len, loan_amnt, int_rate, percent_income, hist_default, cred_hist, grade_num, home_num]])

test_cases = [
    ('🟢 Perfect (Low Risk)', (35, 80000, 10, 5000, 5.0, 0.1, 0, 5, 'A', 'OWN'), 0.2),
    ('🟢 Good (Low Risk)', (40, 60000, 8, 10000, 7.0, 0.15, 0, 3, 'B', 'OWN'), 0.3),
    ('🟡 Moderate Risk', (30, 40000, 3, 15000, 12.0, 0.3, 1, 2, 'C', 'RENT'), 0.5),
    ('🟠 High Risk', (25, 30000, 1, 20000, 18.0, 0.4, 2, 1, 'D', 'RENT'), 0.6),
    ('🔴 Worst (Very High Risk)', (22, 20000, 0, 25000, 25.0, 0.5, 3, 0, 'F', 'RENT'), 0.7),
]

correct = 0
for name, params, threshold in test_cases:
    feat = create_test_case(*params)
    pd = model.predict(xgb.DMatrix(feat, feature_names=feature_cols))[0]

    decision = 'APPROVE' if pd < threshold else 'REVIEW' if pd < threshold + 0.2 else 'REJECT'
    risk_level = 'LOW' if pd < 0.3 else 'MODERATE' if pd < 0.6 else 'HIGH'

    expected = 'APPROVE' if risk_level == 'LOW' else 'REVIEW' if risk_level == 'MODERATE' else 'REJECT'
    is_correct = decision == expected
    correct += 1 if is_correct else 0

    print(f"\n{name}:")
    print(f"  PD: {pd:.4f} ({pd*100:.2f}%) → Risk: {risk_level}")
    print(f"  Decision: {decision} (Expected: {expected}) ({'✓' if is_correct else '❌'})")

print(f"\n{'='*60}")
print(f"Test Case Accuracy: {correct/len(test_cases)*100:.1f}%")
print(f"{'='*60}")

if test_auc > 0.70 and correct/len(test_cases) > 0.7:
    print("\n✅ SUCCESS - Model works on real data!")
else:
    print("\n⚠ Model needs improvement - check feature correlations")

print(f"\n{'='*60}")
print("✅ COMPLETE!")
print(f"{'='*60}")


CREDIT UNDERWRITING - TRAIN ON REAL DATA

Loading your dataset...
Shape: (32581, 12)
Columns: ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']
Default rate: 21.8%

Using 10 features: ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'loan_grade_numeric', 'home_ownership_num']
✓ Features: (32581, 10), Labels: (32581,)
  Default rate: 21.8%

Split: Train=23457, Val=2607, Test=6517

Training XGBoost on REAL data...
[0]	train-auc:0.85965	val-auc:0.85898
[20]	train-auc:0.90553	val-auc:0.89982
[40]	train-auc:0.91802	val-auc:0.91204
[60]	train-auc:0.92471	val-auc:0.91646
[80]	train-auc:0.93160	val-auc:0.92199
[99]	train-auc:0.93537	val-auc:0.92415
✓ Val AUC: 0.9241

TEST SET RESULTS
AUC: 0.9237
Pr

/tmp/ipykernel_26850/1121729803.py:82: UserWarning: [06:03:00] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  model.save_model("credit_risk_model.pkl")
